# GDV – Goal Build-up Analysis FIFA World Cup 2022

## Research question

How are goals created at the FIFA World Cup 2022: through quick attacks with few passes or through longer passing sequences?

In addition, this analysis explores whether a team’s goal build-up style relates to finishing efficiency and tournament progress.

## Use case

A coach, analyst or football fan wants to understand how goals are created. The focus is not only on who scored, but on how the ball was moved before the goal.

The analysis answers the following questions:

1. How many successful passes happen before a goal?
2. Are goals more often created through quick, medium or long build-ups?
3. Which teams score after more direct attacks and which teams score after longer passing sequences?
4. Are more passes before a goal also linked to longer attack duration?
5. Is there a visible relationship between finishing efficiency and tournament progress?
6. What does a concrete Spain goal build-up look like as a case study?

## Dataset

The analysis uses StatsBomb Open Data for the FIFA World Cup 2022. The data contains event-level football actions such as passes, shots, goals, possessions and pitch coordinates.

For each goal, the same possession phase is analysed. Within this possession, successful passes before the goal are counted.

## Limitation

The analysis is based on event data, not tracking data. This means that the ball actions are visible, but not the full movement of all players.

## 1. Setup

This section loads the required Python packages and defines the project paths. The output figures are saved in `reports/figures`, and the processed data is saved in `data/processed`.

In [ ]:
from pathlib import Path
import ast
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsbombpy import sb

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

GOAL_BUILDUPS_FILE = PROCESSED_DIR / "goal_buildups.csv"
GOAL_PASSES_FILE = PROCESSED_DIR / "goal_buildup_passes.csv"

COMPETITION_ID = 43
SEASON_ID = 106

print(PROJECT_ROOT)

## 2. Helper functions

StatsBomb stores pitch locations as coordinate pairs. These helper functions extract x- and y-coordinates and calculate event time in seconds.

In [ ]:
def parse_location(value):
    if isinstance(value, list) and len(value) >= 2:
        return value

    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list) and len(parsed) >= 2:
                return parsed
        except Exception:
            return None

    return None


def get_x(value):
    loc = parse_location(value)
    if isinstance(loc, list):
        return loc[0]
    return np.nan


def get_y(value):
    loc = parse_location(value)
    if isinstance(loc, list):
        return loc[1]
    return np.nan


def seconds_total(row):
    minute = int(row["minute"])
    second = int(row["second"]) if not pd.isna(row["second"]) else 0
    return minute * 60 + second

## 3. Data preprocessing

For each goal, I identify the possession that directly led to the goal. Then I count all successful passes by the scoring team within the same possession before the shot.

This follows the project logic: if possession changes or an interruption occurs, a new possession starts and the build-up sequence also starts again.

In [ ]:
def build_goal_sequences(force_rebuild=False):
    if GOAL_BUILDUPS_FILE.exists() and GOAL_PASSES_FILE.exists() and not force_rebuild:
        buildups = pd.read_csv(GOAL_BUILDUPS_FILE)
        passes = pd.read_csv(GOAL_PASSES_FILE)
        return buildups, passes

    matches = sb.matches(
        competition_id=COMPETITION_ID,
        season_id=SEASON_ID
    )

    all_buildups = []
    all_passes = []

    for match_idx, match in matches.reset_index(drop=True).iterrows():
        match_id = int(match["match_id"])
        home_team = match["home_team"]
        away_team = match["away_team"]
        match_label = f"{home_team} vs {away_team}"

        print(f"{match_idx + 1}/{len(matches)} {match_label}")

        try:
            events = sb.events(match_id=match_id)
        except Exception as exc:
            print(f"Match {match_id} konnte nicht geladen werden: {exc}")
            continue

        needed_columns = [
            "id",
            "index",
            "period",
            "timestamp",
            "minute",
            "second",
            "team",
            "player",
            "type",
            "possession",
            "location",
            "pass_end_location",
            "pass_outcome",
            "shot_outcome",
            "play_pattern"
        ]

        for col in needed_columns:
            if col not in events.columns:
                events[col] = np.nan

        events = events.sort_values(["period", "index"]).copy()

        events["event_seconds"] = events.apply(seconds_total, axis=1)
        events["x"] = events["location"].apply(get_x)
        events["y"] = events["location"].apply(get_y)
        events["end_x"] = events["pass_end_location"].apply(get_x)
        events["end_y"] = events["pass_end_location"].apply(get_y)

        goals = events[
            (events["type"] == "Shot")
            & (events["shot_outcome"].astype(str).str.lower() == "goal")
        ].copy()

        for goal_number, goal in goals.reset_index(drop=True).iterrows():
            team = goal["team"]
            possession = goal["possession"]
            goal_index = goal["index"]
            goal_seconds = goal["event_seconds"]
            build_up_id = f"{match_id}_{goal_number + 1}"

            same_possession = events[
                (events["possession"] == possession)
                & (events["team"] == team)
                & (events["index"] <= goal_index)
            ].copy()

            if same_possession.empty:
                continue

            successful_passes = same_possession[
                (same_possession["type"] == "Pass")
                & (
                    same_possession["pass_outcome"].isna()
                    | (same_possession["pass_outcome"].astype(str).str.lower() == "nan")
                )
            ].copy()

            if successful_passes.empty:
                first_event = same_possession.iloc[0]
                start_x = first_event["x"]
                start_y = first_event["y"]
                start_seconds = first_event["event_seconds"]
            else:
                first_pass = successful_passes.iloc[0]
                start_x = first_pass["x"]
                start_y = first_pass["y"]
                start_seconds = first_pass["event_seconds"]

            duration_seconds = max(0, goal_seconds - start_seconds)

            all_buildups.append(
                {
                    "build_up_id": build_up_id,
                    "match_id": match_id,
                    "match_date": match.get("match_date"),
                    "match_label": match_label,
                    "team": team,
                    "goal_player": goal.get("player"),
                    "goal_minute": goal.get("minute"),
                    "goal_second": goal.get("second"),
                    "period": goal.get("period"),
                    "possession": possession,
                    "play_pattern": goal.get("play_pattern"),
                    "passes_before_goal": len(successful_passes),
                    "duration_seconds": duration_seconds,
                    "start_x": start_x,
                    "start_y": start_y,
                    "goal_x": goal.get("x"),
                    "goal_y": goal.get("y")
                }
            )

            for pass_order, p in successful_passes.reset_index(drop=True).iterrows():
                all_passes.append(
                    {
                        "build_up_id": build_up_id,
                        "match_id": match_id,
                        "match_label": match_label,
                        "team": team,
                        "goal_player": goal.get("player"),
                        "goal_minute": goal.get("minute"),
                        "pass_order": pass_order + 1,
                        "player": p.get("player"),
                        "minute": p.get("minute"),
                        "second": p.get("second"),
                        "x": p.get("x"),
                        "y": p.get("y"),
                        "end_x": p.get("end_x"),
                        "end_y": p.get("end_y")
                    }
                )

        time.sleep(0.1)

    buildups = pd.DataFrame(all_buildups)
    passes = pd.DataFrame(all_passes)

    buildups.to_csv(GOAL_BUILDUPS_FILE, index=False)
    passes.to_csv(GOAL_PASSES_FILE, index=False)

    return buildups, passes

## 4. Load or create the goal build-up dataset

The processed dataset contains one row per goal and one row per successful pass in goal build-ups. If the CSV files already exist, they are loaded directly. Otherwise, they are created from the StatsBomb event data.

In [ ]:
buildups, passes = build_goal_sequences(force_rebuild=False)

print("Goals analysed:", len(buildups))
print("Passes before goals:", len(passes))
print("Teams:", buildups["team"].nunique())

display(buildups.head())
display(passes.head())

## 5. Build-up categories

To make the analysis easier to interpret, goals are grouped into three build-up types:

- Quick: 0 to 3 successful passes before the goal
- Medium: 4 to 7 successful passes before the goal
- Long: 8 or more successful passes before the goal

In [ ]:
def classify_buildup(pass_count):
    if pass_count <= 3:
        return "Quick"
    if pass_count <= 7:
        return "Medium"
    return "Long"


buildups = buildups.copy()
passes = passes.copy()

buildups["build_up_type"] = buildups["passes_before_goal"].apply(classify_buildup)

if "build_up_type" in passes.columns:
    passes = passes.drop(columns=["build_up_type"])

passes = passes.merge(
    buildups[["build_up_id", "build_up_type"]],
    on="build_up_id",
    how="left"
)

display(buildups["build_up_type"].value_counts())
display(buildups[["passes_before_goal", "duration_seconds"]].describe())

## 6. Pitch drawing function

The following function draws a simple football pitch using the StatsBomb coordinate system. It is used for the spatial visualizations and the Spain case study.

In [ ]:
def draw_pitch(ax):
    ax.plot([0, 120], [0, 0], color="black", linewidth=1)
    ax.plot([0, 120], [80, 80], color="black", linewidth=1)
    ax.plot([0, 0], [0, 80], color="black", linewidth=1)
    ax.plot([120, 120], [0, 80], color="black", linewidth=1)

    ax.plot([60, 60], [0, 80], color="black", linewidth=1)
    ax.add_patch(plt.Circle((60, 40), 10, fill=False, color="black", linewidth=1))

    ax.plot([0, 18], [18, 18], color="black", linewidth=1)
    ax.plot([18, 18], [18, 62], color="black", linewidth=1)
    ax.plot([18, 0], [62, 62], color="black", linewidth=1)

    ax.plot([120, 102], [18, 18], color="black", linewidth=1)
    ax.plot([102, 102], [18, 62], color="black", linewidth=1)
    ax.plot([102, 120], [62, 62], color="black", linewidth=1)

    ax.set_xlim(0, 120)
    ax.set_ylim(80, 0)
    ax.set_aspect("equal")
    ax.axis("off")

## 7. Static visual analysis

The following figures answer the main research question step by step. The goal is not to create many plots, but to build a clear argument about how goals are created.

### Figure 1: Passes before goals

This histogram shows the distribution of successful passes before goals. It directly answers whether goals are usually created after few passes or after longer passing sequences.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

max_passes = int(buildups["passes_before_goal"].max())

ax.hist(
    buildups["passes_before_goal"],
    bins=range(0, max_passes + 2),
    edgecolor="black"
)

ax.set_title("Passes before goals at the FIFA World Cup 2022")
ax.set_xlabel("Successful passes before goal")
ax.set_ylabel("Number of goals")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "goal_fig01_passes_before_goals.png", dpi=300, bbox_inches="tight")
plt.show()

The histogram is useful for exploration because it shows the full distribution. However, for communication it is still quite detailed. Therefore, the next figure groups goals into quick, medium and long build-ups.

### Figure 2: Build-up types

This figure simplifies the distribution into three categories. It makes the main pattern easier to read than the histogram.

In [ ]:
build_up_counts = (
    buildups["build_up_type"]
    .value_counts()
    .reindex(["Quick", "Medium", "Long"])
)

fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    build_up_counts.index,
    build_up_counts.values
)

ax.set_title("Goal build-up types")
ax.set_xlabel("Build-up type")
ax.set_ylabel("Number of goals")

for i, value in enumerate(build_up_counts.values):
    ax.text(i, value + 0.5, str(int(value)), ha="center")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "goal_fig02_buildup_types.png", dpi=300, bbox_inches="tight")
plt.show()

display(build_up_counts)

### Figure 3: Team comparison

This figure compares the average number of passes before goals by team. Teams with lower values score more directly, while teams with higher values score after longer passing sequences.

Only teams with at least two goals are included to reduce the effect of single-goal outliers.

In [ ]:
team_goal_summary = (
    buildups.groupby("team")
    .agg(
        goals=("build_up_id", "count"),
        avg_passes=("passes_before_goal", "mean"),
        median_passes=("passes_before_goal", "median"),
        avg_duration=("duration_seconds", "mean")
    )
    .reset_index()
)

team_goal_summary_filtered = (
    team_goal_summary[team_goal_summary["goals"] >= 2]
    .sort_values("avg_passes", ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 6))

ax.barh(
    team_goal_summary_filtered["team"],
    team_goal_summary_filtered["avg_passes"]
)

ax.set_title("Average passes before goal by team")
ax.set_xlabel("Average successful passes before goal")
ax.set_ylabel("Team")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "goal_fig03_avg_passes_by_team.png", dpi=300, bbox_inches="tight")
plt.show()

display(team_goal_summary_filtered)

### Figure 4: Passes and attack duration

This scatterplot checks whether more passes before a goal also mean a longer attack duration. This is important because pass count alone does not fully describe the speed of an attack.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for build_type, group in buildups.groupby("build_up_type"):
    ax.scatter(
        group["passes_before_goal"],
        group["duration_seconds"],
        label=build_type,
        alpha=0.75,
        s=55
    )

ax.set_title("Passes before goal vs build-up duration")
ax.set_xlabel("Successful passes before goal")
ax.set_ylabel("Build-up duration in seconds")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURE_DIR / "goal_fig04_passes_vs_duration.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Exploratory spatial plots

Start-location heatmaps and raw pass-flow maps were tested during exploration. They were visually interesting, but not used as main final figures because the patterns were difficult to interpret without a comparison to non-goal attacks.

For the final GDV story, the analysis focuses on stronger and more interpretable variables: pass count, build-up duration, finishing efficiency, tournament progress and one concrete Spain case study.


## 9. Team efficiency and tournament progress

Pass count alone does not explain team success. Therefore, I add team-level shot information and tournament stage information. This allows a comparison between goal build-up style, finishing efficiency and tournament progress.

In [ ]:
TEAM_CONTEXT_FILE = PROCESSED_DIR / "team_tournament_context.csv"

def build_team_context(force_rebuild=False):
    if TEAM_CONTEXT_FILE.exists() and not force_rebuild:
        return pd.read_csv(TEAM_CONTEXT_FILE)

    matches = sb.matches(
        competition_id=COMPETITION_ID,
        season_id=SEASON_ID
    )

    stage_order = {
        "Group Stage": 1,
        "Round of 16": 2,
        "Quarter-finals": 3,
        "Semi-finals": 4,
        "3rd Place Final": 5,
        "Final": 6
    }

    all_shots = []
    all_stages = []

    for _, match in matches.reset_index(drop=True).iterrows():
        match_id = int(match["match_id"])
        home_team = match["home_team"]
        away_team = match["away_team"]
        stage = str(match.get("competition_stage"))

        all_stages.append({"team": home_team, "stage": stage})
        all_stages.append({"team": away_team, "stage": stage})

        try:
            events = sb.events(match_id=match_id)
        except Exception:
            continue

        if "type" not in events.columns:
            continue

        for col in ["team", "shot_outcome", "possession"]:
            if col not in events.columns:
                events[col] = np.nan

        shots_df = events[events["type"] == "Shot"].copy()

        for _, shot in shots_df.iterrows():
            all_shots.append(
                {
                    "team": shot["team"],
                    "shot_outcome": shot.get("shot_outcome"),
                    "possession": shot.get("possession")
                }
            )

    shots_table = pd.DataFrame(all_shots)
    stages_table = pd.DataFrame(all_stages)

    team_shots = (
        shots_table.groupby("team")
        .agg(
            shots=("team", "count"),
            shot_possessions=("possession", "nunique")
        )
        .reset_index()
    )

    goals_table = shots_table[
        shots_table["shot_outcome"].astype(str).str.lower() == "goal"
    ]

    team_goals = (
        goals_table.groupby("team")
        .size()
        .reset_index(name="goals")
    )

    team_stage = stages_table.copy()
    team_stage["stage_score"] = team_stage["stage"].map(stage_order).fillna(0)

    team_stage = (
        team_stage.sort_values("stage_score")
        .groupby("team")
        .tail(1)
        .reset_index(drop=True)
    )

    context = team_shots.merge(team_goals, on="team", how="left")
    context = context.merge(team_stage[["team", "stage", "stage_score"]], on="team", how="left")

    context["goals"] = context["goals"].fillna(0)
    context["conversion_rate"] = context["goals"] / context["shots"]
    context["goals_per_shot_possession"] = context["goals"] / context["shot_possessions"]

    context.to_csv(TEAM_CONTEXT_FILE, index=False)

    return context

team_context = build_team_context(force_rebuild=False)

display(team_context.sort_values("stage_score", ascending=False).head(10))

## 10. Combined team analysis table

This table combines build-up information with finishing efficiency and tournament stage. It is the basis for the following team-level figures.

In [ ]:
team_buildup = (
    buildups.groupby("team")
    .agg(
        goals_in_buildup_data=("build_up_id", "count"),
        avg_passes_before_goal=("passes_before_goal", "mean"),
        median_passes_before_goal=("passes_before_goal", "median"),
        avg_duration_seconds=("duration_seconds", "mean")
    )
    .reset_index()
)

team_analysis = team_buildup.merge(
    team_context,
    on="team",
    how="left"
)

team_analysis = team_analysis[team_analysis["goals_in_buildup_data"] >= 2].copy()

display(
    team_analysis.sort_values("stage_score", ascending=False)
)

## 11. Final GDV figures

The following plots are the final design iteration. They are more useful than the earlier heatmaps because they directly connect the research question to measurable outcomes: build-up style, efficiency and tournament progress.


### Figure 5: Team build-up style before goals

This plot compares teams by average passes before goal and average build-up duration. The color shows finishing efficiency and the point size represents tournament progress.

It helps answer whether successful teams tend to score through more direct or more patient attacks.


In [ ]:
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "#333333"
plt.rcParams["axes.labelcolor"] = "#222222"
plt.rcParams["xtick.color"] = "#222222"
plt.rcParams["ytick.color"] = "#222222"
plt.rcParams["font.size"] = 10

stage_labels = {
    1: "Group",
    2: "R16",
    3: "QF",
    4: "SF",
    5: "3rd/Final",
    6: "Final"
}

plot_data = team_analysis.copy()

fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(
    plot_data["avg_passes_before_goal"],
    plot_data["avg_duration_seconds"],
    s=plot_data["stage_score"] * 90,
    c=plot_data["conversion_rate"],
    cmap="viridis",
    alpha=0.8,
    edgecolor="black",
    linewidth=0.7
)

for _, row in plot_data.iterrows():
    ax.text(
        row["avg_passes_before_goal"] + 0.05,
        row["avg_duration_seconds"] + 0.25,
        row["team"],
        fontsize=8
    )

ax.set_title("Team build-up style before goals")
ax.set_xlabel("Average successful passes before goal")
ax.set_ylabel("Average build-up duration in seconds")

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Conversion rate")

ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "goal_final_01_team_buildup_style.png", dpi=300, bbox_inches="tight")
plt.show()


### Figure 6: Finishing efficiency and tournament progress

This plot checks whether teams with higher finishing efficiency tended to progress further in the tournament. It shows association, not causation.

The point size represents the number of goals and the color represents the average number of passes before goal.


In [ ]:
plot_data = team_analysis.copy()

fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(
    plot_data["conversion_rate"],
    plot_data["stage_score"],
    s=plot_data["goals"] * 55,
    c=plot_data["avg_passes_before_goal"],
    cmap="plasma",
    alpha=0.8,
    edgecolor="black",
    linewidth=0.7
)

for _, row in plot_data.iterrows():
    ax.text(
        row["conversion_rate"] + 0.003,
        row["stage_score"] + 0.03,
        row["team"],
        fontsize=8
    )

ax.set_title("Finishing efficiency and tournament progress")
ax.set_xlabel("Conversion rate")
ax.set_ylabel("Tournament stage reached")

ax.set_yticks([1, 2, 3, 4, 5, 6])
ax.set_yticklabels([stage_labels[i] for i in [1, 2, 3, 4, 5, 6]])

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Average passes before goal")

ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "goal_final_02_efficiency_stage.png", dpi=300, bbox_inches="tight")
plt.show()


### Figure 7: Direct vs patient goal build-ups

This bar chart ranks teams by average passes before goal. It is easy to explain because teams on the left are more direct and teams on the right are more patient in their goal build-ups.

Teams with fewer than two goals are excluded to reduce the influence of single-goal outliers.


In [ ]:
team_rank = team_analysis.sort_values("avg_passes_before_goal").copy()

fig, ax = plt.subplots(figsize=(10, 7))

bars = ax.barh(
    team_rank["team"],
    team_rank["avg_passes_before_goal"],
    color="#4C78A8"
)

ax.axvline(
    team_analysis["avg_passes_before_goal"].mean(),
    color="#E45756",
    linestyle="--",
    linewidth=2,
    label="Average"
)

for bar, (_, row) in zip(bars, team_rank.iterrows()):
    ax.text(
        bar.get_width() + 0.05,
        bar.get_y() + bar.get_height() / 2,
        f"{int(row['goals'])} goals",
        va="center",
        fontsize=8
    )

ax.set_title("Direct vs patient goal build-ups by team")
ax.set_xlabel("Average successful passes before goal")
ax.set_ylabel("Team")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURE_DIR / "goal_final_03_direct_vs_patient.png", dpi=300, bbox_inches="tight")
plt.show()


### Figure 8: Share of goal build-up types by team

This plot shows whether a team's goals came mainly from quick, medium or long build-ups. It is more informative than only showing the average because it reveals the mix of attacking patterns.


In [ ]:
team_type_counts = (
    buildups.groupby(["team", "build_up_type"])
    .size()
    .reset_index(name="count")
)

team_type_counts = team_type_counts.merge(
    team_analysis[["team", "goals"]],
    on="team",
    how="inner"
)

team_type_counts["share"] = team_type_counts["count"] / team_type_counts["goals"]

team_type_pivot = (
    team_type_counts.pivot_table(
        index="team",
        columns="build_up_type",
        values="share",
        fill_value=0
    )
)

team_type_pivot = team_type_pivot.reindex(
    team_analysis.sort_values("stage_score", ascending=False)["team"]
)

for col in ["Quick", "Medium", "Long"]:
    if col not in team_type_pivot.columns:
        team_type_pivot[col] = 0

team_type_pivot = team_type_pivot[["Quick", "Medium", "Long"]]

fig, ax = plt.subplots(figsize=(10, 7))

team_type_pivot.plot(
    kind="barh",
    stacked=True,
    ax=ax,
    color=["#2ca25f", "#fdae61", "#de2d26"]
)

ax.set_title("Share of goal build-up types by team")
ax.set_xlabel("Share of goals")
ax.set_ylabel("Team")
ax.legend(title="Build-up type", loc="lower right")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "goal_final_04_buildup_type_share.png", dpi=300, bbox_inches="tight")
plt.show()


## 12. Spain case study

Spain is used as a case study because Spain is strongly associated with possession-based football. A Spanish goal with many passes is therefore a plausible example for showing a longer passing sequence before a goal.


In [ ]:
spain_goals = buildups[buildups["team"] == "Spain"].copy()

display(
    spain_goals[
        ["build_up_id", "match_label", "goal_player", "goal_minute", "passes_before_goal", "duration_seconds"]
    ].sort_values("passes_before_goal", ascending=False)
)

### Figure 9: Spain goal build-up sequence

This figure shows one concrete goal possession from Spain. The numbers show the order of the successful passes before the goal.

This figure is useful for communication because it turns the aggregate analysis into one understandable football sequence.


In [ ]:
case_goal = spain_goals.sort_values("passes_before_goal", ascending=False).iloc[0]
case_id = case_goal["build_up_id"]

case_passes = (
    passes[passes["build_up_id"] == case_id]
    .sort_values("pass_order")
)

fig, ax = plt.subplots(figsize=(10, 6))

draw_pitch(ax)

for _, p in case_passes.iterrows():
    ax.arrow(
        p["x"],
        p["y"],
        p["end_x"] - p["x"],
        p["end_y"] - p["y"],
        length_includes_head=True,
        head_width=1.5,
        alpha=0.75
    )

    ax.text(
        p["x"],
        p["y"],
        str(int(p["pass_order"])),
        fontsize=9,
        weight="bold"
    )

ax.scatter(
    [case_goal["goal_x"]],
    [case_goal["goal_y"]],
    marker="*",
    s=350,
    edgecolor="black",
    label="Goal"
)

ax.set_title(
    f"Spain case study: goal by {case_goal['goal_player']} after {case_goal['passes_before_goal']} passes"
)

ax.legend(loc="lower center")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "goal_fig13_spain_case_study.png", dpi=300, bbox_inches="tight")
plt.show()

display(case_passes[["pass_order", "player", "minute", "second", "x", "y", "end_x", "end_y"]])


## 13. Summary

This section summarizes the main numerical findings and lists the final figures used for the GDV report.


In [ ]:
total_goals = len(buildups)
avg_passes = buildups["passes_before_goal"].mean()
median_passes = buildups["passes_before_goal"].median()
avg_duration = buildups["duration_seconds"].mean()

quick_goals = (buildups["build_up_type"] == "Quick").sum()
medium_goals = (buildups["build_up_type"] == "Medium").sum()
long_goals = (buildups["build_up_type"] == "Long").sum()

quick_share = quick_goals / total_goals
medium_share = medium_goals / total_goals
long_share = long_goals / total_goals

print("GDV summary")
print("-----------")
print(f"Analysed goals: {total_goals}")
print(f"Average passes before goal: {avg_passes:.2f}")
print(f"Median passes before goal: {median_passes:.2f}")
print(f"Average build-up duration: {avg_duration:.2f} seconds")
print(f"Quick goals: {quick_goals} ({quick_share:.1%})")
print(f"Medium goals: {medium_goals} ({medium_share:.1%})")
print(f"Long goals: {long_goals} ({long_share:.1%})")

print()
print("Final figures for the GDV report:")
final_figures = [
    "goal_fig01_passes_before_goals.png",
    "goal_fig02_buildup_types.png",
    "goal_fig04_passes_vs_duration.png",
    "goal_final_01_team_buildup_style.png",
    "goal_final_02_efficiency_stage.png",
    "goal_final_03_direct_vs_patient.png",
    "goal_final_04_buildup_type_share.png",
    "goal_fig13_spain_case_study.png"
]

for name in final_figures:
    print(FIGURE_DIR / name)

print()
print("Highest average passes before goal:")
display(team_analysis.sort_values("avg_passes_before_goal", ascending=False).head(5)[
    ["team", "goals", "avg_passes_before_goal", "avg_duration_seconds", "conversion_rate", "stage"]
])

print("Highest conversion rate:")
display(team_analysis.sort_values("conversion_rate", ascending=False).head(5)[
    ["team", "goals", "shots", "conversion_rate", "avg_passes_before_goal", "stage"]
])


## 14. Design reflection

The first exploration included heatmaps and raw pass-flow maps. These were useful for understanding the data, but they were not strong enough as final GDV figures because the spatial patterns were difficult to interpret without a baseline of non-goal attacks.

The final design focuses on plots with clearer analytical value:

- distributions and categories for the main question,
- scatterplots for relationships,
- bar charts for team comparison,
- and one Spain case study for a concrete football example.

This follows the visualization pipeline: explore, design, evaluate critically, iterate and communicate.


## 14. Limitations

The analysis uses event data, not tracking data. Therefore, it shows ball actions but not the movement of all players.

The possession ID is used as an approximation for the attacking sequence before a goal. This is reasonable for the project, but it is not a complete tactical reconstruction.

Finishing efficiency and tournament progress are compared descriptively. The figures show possible patterns, but they do not prove causality.

Teams with few goals can have unstable averages. For team-level comparisons, teams with at least two goals are used to reduce the impact of single-goal outliers.